# Programowanie w języku Python

## Wykład 14 — Współbieżność: wątki, procesy i asyncio

### dr inż. Waldemar Bauer

## Cel wykładu

Po dzisiejszych zajęciach powinieneś:

- odróżniać współbieżność od równoległości,
- rozumieć różnicę między zadaniami I/O-bound i CPU-bound,
- znać rolę procesu i wątku,
- umieć uruchomić prosty wątek,
- znać `ThreadPoolExecutor`,
- rozumieć race condition i potrzebę synchronizacji,
- znać podstawowe zastosowanie `Lock` i `Queue`,
- znać `ProcessPoolExecutor`,
- rozumieć ideę event loop i `asyncio`,
- umieć dobrać model współbieżności do problemu.

## Dlaczego współbieżność?

Program często czeka:

```text
na plik
na sieć
na użytkownika
na urządzenie
na bazę danych
```

W czasie oczekiwania może być możliwe wykonywanie innego zadania.

## Współbieżność a równoległość

```text
WSPÓŁBIEŻNOŚĆ
↓
wiele zadań robi postęp w tym samym okresie

RÓWNOLEGŁOŚĆ
↓
wiele zadań rzeczywiście wykonuje się jednocześnie
```

To nie są synonimy.

## Dwa typowe rodzaje obciążenia

```text
I/O-bound
↓
większość czasu czekamy na wejście/wyjście

CPU-bound
↓
większość czasu wykonujemy obliczenia
```

Ten podział pomaga dobrać narzędzie.

## Przykład I/O-bound

```text
pobierz plik
↓
czekaj na sieć

pobierz drugi plik
↓
czekaj na sieć
```

Jeżeli zadania mogą czekać niezależnie, współbieżność może skrócić całkowity czas.

## Przykład CPU-bound

```text
FFT
symulacja
kompresja
duża pętla numeryczna
```

Tutaj problemem jest czas procesora, a nie oczekiwanie na I/O.

## Proces

Proces to uruchomiony program z własną przestrzenią pamięci.

Dwa procesy są od siebie mocniej odizolowane niż dwa wątki.

Wymiana danych między procesami wymaga dodatkowego mechanizmu.

## Wątek

Wątki działają wewnątrz jednego procesu.

Współdzielą pamięć procesu.

To ułatwia współpracę, ale tworzy też ryzyko:

> kilka wątków może jednocześnie modyfikować ten sam stan.

## Pierwszy wątek

Moduł standardowy:

```python
import threading
```

In [ ]:
import threading
import time


def worker(name):
    print(f"{name}: start")
    time.sleep(1)
    print(f"{name}: stop")


thread = threading.Thread(
    target=worker,
    args=("worker-1",),
)

thread.start()
thread.join()

## `start()` i `join()`

```python
thread.start()
```

uruchamia zadanie w wątku.

```python
thread.join()
```

oznacza:

> poczekaj, aż ten wątek zakończy pracę.

## Dwa zadania I/O-like

`time.sleep()` symuluje oczekiwanie.

In [ ]:
def wait_task(name, seconds):
    print(f"{name}: start")
    time.sleep(seconds)
    print(f"{name}: stop")


start = time.perf_counter()

threads = [
    threading.Thread(
        target=wait_task,
        args=(f"T{i}", 1),
    )
    for i in range(3)
]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

print(
    f"Czas: {time.perf_counter() - start:.2f} s"
)

## Nie zapamiętujemy konkretnego czasu

Wynik zależy od systemu i obciążenia.

Interesuje nas mechanizm:

```text
zadanie A czeka
↓
inne zadanie może działać
```

To właśnie dobre środowisko dla współbieżności I/O.

## ThreadPoolExecutor

W praktycznym kodzie często wygodniejszy od ręcznego zarządzania wątkami jest:

```python
concurrent.futures.ThreadPoolExecutor
```

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def fetch_like_task(value):
    time.sleep(0.5)
    return value * 2


with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(
        executor.map(
            fetch_like_task,
            range(8),
        )
    )

print(results)

## Co daje executor?

Executor zarządza pulą pracowników.

My opisujemy:

```text
jakie zadanie wykonać
↓
dla jakich danych
```

a executor zarządza uruchamianiem i oczekiwaniem.

## Współdzielona pamięć to także problem

Rozważmy wspólny licznik.

Kilka wątków może wykonywać operację:

```python
counter += 1
```

w podobnym czasie.

Operacja logicznie wygląda jak jedna instrukcja, ale może składać się z wielu kroków.

## Race condition

Race condition występuje, gdy wynik zależy od kolejności współbieżnych operacji.

```text
wątek A: odczytaj counter
wątek B: odczytaj counter
wątek A: zapisz counter + 1
wątek B: zapisz counter + 1
```

Jedna aktualizacja może zostać utracona.

## Nie polegamy na „przypadkowym bezpieczeństwie”

To, że prosty eksperyment kilka razy daje poprawny wynik, nie dowodzi bezpieczeństwa współbieżnego kodu.

Synchronizacja powinna wynikać z projektu, a nie z obserwacji „u mnie działa”.

## Lock

`threading.Lock` pozwala chronić sekcję krytyczną.

In [ ]:
lock = threading.Lock()
counter = 0


def increment_many(times):
    global counter

    for _ in range(times):
        with lock:
            counter += 1


threads = [
    threading.Thread(
        target=increment_many,
        args=(10_000,),
    )
    for _ in range(4)
]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

print(counter)

## Sekcja krytyczna

To część kodu, która korzysta ze współdzielonego zasobu i nie może być bezpiecznie wykonywana jednocześnie przez wielu pracowników.

```python
with lock:
    ...
```

powinna być możliwie krótka.

## Zbyt dużo blokad też jest problemem

Blokady mogą prowadzić do:

- spadku wydajności,
- trudnego debugowania,
- zakleszczeń.

Dlatego często lepiej ograniczyć współdzielony stan zamiast „obudowywać wszystko Lockiem”.

## Queue — przekazywanie danych

Dla modelu producent–konsument bardzo przydatna jest:

```python
queue.Queue
```

Kolejka jest zaprojektowana do bezpiecznej komunikacji między wątkami.

In [ ]:
from queue import Queue


queue = Queue()


def producer():
    for value in range(5):
        queue.put(value)

    queue.put(None)


def consumer():
    while True:
        value = queue.get()

        if value is None:
            break

        print("Odebrano:", value)


producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

producer_thread.start()
consumer_thread.start()

producer_thread.join()
consumer_thread.join()

## Producent i konsument

```text
PRODUCENT
↓
Queue
↓
KONSUMENT
```

Zamiast współdzielić ręcznie listę i synchronizować jej modyfikacje, przekazujemy dane przez kontrolowany kanał.

## GIL — ważny kontekst CPythona

W klasycznym buildzie CPythona wykonywanie kodu Python w wielu wątkach jest ograniczane przez Global Interpreter Lock.

Praktyczny wniosek:

> wątki są bardzo użyteczne dla I/O-bound, ale nie zakładamy automatycznego przyspieszenia czystych obliczeń CPU przez dodanie wątków.

## A co z free-threaded Python?

Python rozwija również buildy pozwalające uruchamiać interpreter bez tradycyjnego GIL.

To jednak **nie zmienia podstawowej lekcji projektowej**:

- współdzielony stan nadal wymaga ostrożności,
- kod współbieżny nadal może mieć race conditions,
- narzędzie nadal dobieramy do charakteru zadania.

## CPU-bound: procesy

Dla niezależnych obliczeń CPU można użyć wielu procesów.

Standardowa abstrakcja:

```python
ProcessPoolExecutor
```

Procesy mają osobne interpretery i osobne przestrzenie pamięci.

In [ ]:
from concurrent.futures import ProcessPoolExecutor


def square(value):
    return value * value


# W skrypcie uruchamiamy zwykle pod:
# if __name__ == "__main__":
#     with ProcessPoolExecutor() as executor:
#         print(list(executor.map(square, range(10))))

## Dlaczego komentarz w notebooku?

Uruchamianie procesów zależy od sposobu startowania interpretera i systemu operacyjnego.

W normalnym skrypcie stosujemy:

```python
if __name__ == "__main__":
    ...
```

To wraca do W7.

## Koszt procesów

Procesy dają silniejszą izolację i mogą wykonywać pracę CPU równolegle.

Ale mają koszt:

- utworzenia procesu,
- przesyłania danych,
- serializacji argumentów i wyników,
- większego zużycia pamięci.

Nie są automatycznie lepsze od wątków.

## asyncio — trzeci model

Dla wielu operacji asynchronicznych I/O Python oferuje:

```python
asyncio
```

Zamiast tworzyć wątek na każde zadanie, używamy pętli zdarzeń i zadań, które świadomie oddają sterowanie podczas oczekiwania.

## `async def`

Funkcja:

```python
async def task():
    ...
```

tworzy funkcję asynchroniczną.

Jej wykonaniem zarządza event loop.

In [ ]:
import asyncio


async def task(name):
    print(name, "start")
    await asyncio.sleep(1)
    print(name, "stop")
    return name

## `await`

```python
await operation
```

oznacza w uproszczeniu:

> poczekaj na tę operację, ale pozwól event loop wykonywać w tym czasie inne gotowe zadania.

To współbieżność kooperacyjna.

## Uruchamianie wielu zadań

W zwykłym skrypcie:

In [ ]:
async def main_async():
    results = await asyncio.gather(
        task("A"),
        task("B"),
        task("C"),
    )

    return results


# W notebooku można użyć:
results = await main_async()
print(results)

## Threading a asyncio

```text
THREADING
↓
system zarządza przełączaniem wątków
↓
dobry dla wielu blokujących operacji I/O

ASYNCIO
↓
zadanie oddaje sterowanie przez await
↓
dobry dla bibliotek oferujących asynchroniczne API
```

Nie zamieniamy automatycznie każdej funkcji na `async def`.

## async nie przyspiesza obliczeń CPU

Kod:

```python
async def calculate():
    # długa pętla CPU
```

nie staje się równoległy tylko dlatego, że dodaliśmy `async`.

Jeżeli funkcja nie dochodzi do punktu `await`, może blokować event loop.

## Jak dobrać narzędzie?

```text
JEDNO PROSTE ZADANIE
→ zwykły kod synchroniczny

WIELE BLOKUJĄCYCH OPERACJI I/O
→ ThreadPoolExecutor

WIELE ASYNCHRONICZNYCH OPERACJI I/O
→ asyncio

NIEZALEŻNE CIĘŻKIE OBLICZENIA CPU
→ ProcessPoolExecutor
```

To punkt startowy, nie absolutna reguła.

## Najpierw wersja poprawna

Nie zaczynamy projektu od współbieżności.

Dobra kolejność:

```text
1. poprawny kod synchroniczny
2. pomiar
3. identyfikacja wąskiego gardła
4. wybór modelu
5. ponowny pomiar
```

Współbieżność zwiększa złożoność programu.

## Przykład integrujący: wiele „odczytów”

Załóżmy, że odczyt każdego urządzenia trwa około 0.5 s.

In [ ]:
def read_sensor(sensor_id):
    time.sleep(0.5)

    return {
        "sensor": sensor_id,
        "value": 20 + sensor_id,
    }


sensor_ids = list(range(6))

with ThreadPoolExecutor(max_workers=3) as executor:
    measurements = list(
        executor.map(
            read_sensor,
            sensor_ids,
        )
    )

print(measurements)

## Obsługa wyjątków w executorze

Zadanie współbieżne również może zgłosić wyjątek.

Nie możemy zakładać, że „wątek sam sobie poradzi”.

Przy pracy z `Future` wyjątek zostanie ponownie zgłoszony przy pobieraniu wyniku.

## Czego unikać

- współdzielenia wielu mutowalnych struktur bez kontroli,
- uruchamiania nieograniczonej liczby wątków,
- blokowania event loop ciężką pracą CPU,
- dodawania współbieżności bez pomiaru,
- zakładania, że kolejność wykonania jest deterministyczna,
- traktowania `sleep()` jako mechanizmu synchronizacji.

## Ćwiczenie

Masz funkcję:

```python
read_device(device_id)
```

która symuluje 0.5 s oczekiwania i zwraca słownik.

1. Uruchom ją synchronicznie dla 8 urządzeń.
2. Zmierz czas.
3. Uruchom przez `ThreadPoolExecutor`.
4. Zmierz czas ponownie.
5. Dodaj jedno urządzenie zgłaszające wyjątek.
6. Obsłuż błąd bez ukrywania wyników pozostałych zadań.
7. Wyjaśnij, dlaczego ten przykład jest I/O-bound, a nie CPU-bound.

## Podsumowanie

Po tym wykładzie powinieneś pamiętać:

- współbieżność i równoległość to różne pojęcia,
- model zależy od charakteru obciążenia,
- wątki współdzielą pamięć,
- współdzielony stan może prowadzić do race conditions,
- `Lock` chroni sekcję krytyczną,
- `Queue` pomaga bezpiecznie przekazywać dane,
- `ThreadPoolExecutor` upraszcza zadania I/O,
- procesy dają silniejszą izolację i nadają się do części zadań CPU-bound,
- `asyncio` opiera się na event loop i `await`,
- `async` nie oznacza automatycznie „szybciej”,
- współbieżność dodajemy po uzyskaniu poprawnej wersji i wykonaniu pomiaru.

## Następny wykład

# Od skryptu do aplikacji

Na ostatnim wykładzie połączymy elementy całego kursu w jeden mały projekt i zobaczymy, jak decyzje z W1–W14 składają się na strukturę rzeczywistej aplikacji.